In [0]:
import pyspark.sql.functions as F
from pyspark.sql.functions import col, trim, StringType

#Reading From Silver Table

In [0]:
%sql
SELECT * FROM silver.transtractions

In [0]:
query = """
SELECT 
    T.transaction_id,
    D.date_key,
    PC.promo_key,
    T.order_date,
    S.store_key,
    T.store_id,
    T.store_name,
    T.region,
    P.product_key,
    T.product_id,
    T.product_name,
    T.category,
    T.quantity,
    T.applied_unit_price,
    T.applied_discount,
    T.customer_id,
    T.loyalty_member
FROM silver.transtractions T
LEFT JOIN gold.dim_store S
    ON T.store_id = S.store_id
LEFT JOIN gold.dim_product P
    ON T.product_id = P.product_id
LEFT JOIN gold.dim_promo_calander PC
    ON T.order_date BETWEEN PC.start_date AND PC.end_date
    AND T.store_id = PC.store_id
    AND T.category = PC.category
LEFT JOIN gold.dim_date D
    ON T.order_date = D.full_day
"""

df = spark.sql(query)

In [0]:
df.count()

In [0]:
%sql
SELECT COUNT(*) FROM silver.transtractions

In [0]:
df.display()

In [0]:
from pyspark.sql import functions as F
# Number of nulls
K = df.filter(F.col("promo_key").isNotNull())





In [0]:
K.display()

#Write into gold layer

In [0]:
(
    df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("gold.fact_tanstractions")
)